# Warehouse Liquidation Strategy

This notebook builds the liquidation lists and final report from the inventory data and merchandising memo.

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

def find_data_dir() -> Path:
    candidates = [
        Path("/workspace/data"),
        Path("/workspace/environment/data"),
        Path("../environment/data"),
        Path("environment/data"),
        Path("../data"),
        Path("data"),
    ]

    for p in candidates:
        if p.exists():
            return p.resolve()

    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in ("environment/data", "data"):
            p = base / rel
            if p.exists():
                return p.resolve()

    raise FileNotFoundError(f"Could not locate data directory from {Path.cwd()}")

DATA_DIR = find_data_dir()
WORKSPACE_DIR = DATA_DIR.parents[1]

DATA_PATH = DATA_DIR / "Q3_Inventory_Sales_Data.csv"
MEMO_PATH = DATA_DIR / "merchandising_strategy.txt"

In [30]:
with open(MEMO_PATH, "r", encoding="utf-8") as f:
    memo_text = f.read()

print(memo_text)

df = pd.read_csv(DATA_PATH)
df["Launch_Date"] = pd.to_datetime(df["Launch_Date"])

print("Shape:", df.shape)
display(df.head(8))

To: Inventory Planning Team
From: VP of Merchandising
Date: September 26, 2024
Subject: Q4 Warehouse Capacity & Liquidation Strategy

Team,

We need to free up physical storage space ahead of the holiday inbound wave.

Please review Q3 inventory and sales, then propose liquidation recommendations. Keep the following guardrails in mind:

- Protect the Home Automation category. Do not liquidate any Home Automation SKU.
- Do not liquidate any SKU launched within the last 8 weeks. They have not had enough time to mature.
- Some weekly sales were distorted by one-off B2B bulk transfers. Normalize those anomalies before comparing item performance.
- If you liquidate a base SKU, you must also liquidate any dependent accessories or add-ons that require it. We cannot leave orphaned components behind.
- Damaged SKUs should be prioritized first because they are liabilities.
- Any SKU that would take more than two years to deplete at its normalized sales rate should be treated as dead stock.

Than

,SKU,Product_Name,Category,Condition,Launch_Date,Inventory_Qty,Unit_Volume,Unit_Margin,Requires_Base_SKU,Week_1,Week_2,Week_3,Week_4,Week_5,Week_6,Week_7,Week_8,Week_9,Week_10,Week_11,Week_12
0,AUD126,Audio Buds 1,Audio,New,2024-07-18,202,0.83,5.02,NaN,0,7,10,10,4,7,11,11,7,10,12,12
1,AUD127,Audio Buds 2,Audio,New,2024-07-30,189,2.16,30.17,NaN,0,0,35,29,34,35,23,16,40,31,16,24
2,AUD128,Audio Buds 3,Audio,New,2024-07-16,191,1.72,30.56,NaN,4,4,5,5,4,5,4,4,4,4,5,5
3,AUD129,Audio Buds 4,Audio,New,2024-07-28,898,0.49,14.91,NaN,0,0,20,23,19,29,17,20,15,17,20,22
4,AUD130,Audio Buds 5,Audio,New,2024-07-19,1821,2.61,7.57,NaN,0,15,15,14,23,17,19,18,22,15,14,20
5,AUD131,Audio Buds 6,Audio,New,2024-07-16,813,1.28,8.85,NaN,6,5,5,6,7,2,3,4,3,3,4,3
6,AUD132,Audio Buds 7,Audio,New,2024-07-28,1988,0.26,5.20,NaN,0,0,33,23,11,16,12,21,21,19,32,24
7,AUD133,Audio Buds 8,Audio,Damaged,2024-07-24,1111,0.77,15.96,NaN,0,1,1,1,0,1,1,1,2,2,1,1


In [31]:
from scipy.optimize import milp, LinearConstraint, Bounds

SALES_COLUMNS = [f"Week_{i}" for i in range(1, 13)]
ANCHOR_DATE = pd.Timestamp("2024-09-26")
RECENCY_CUTOFF = ANCHOR_DATE - pd.Timedelta(weeks=8)


## Normalize sales and apply business rules

In [32]:
def normalized_avg_weekly_sales(row: pd.Series) -> float:
    sales = row[SALES_COLUMNS].to_numpy(dtype=float)

    weeks_since_launch = max(1, (ANCHOR_DATE - row["Launch_Date"]).days // 7)
    weeks_active = min(12, weeks_since_launch)

    active_sales = sales[-weeks_active:]

    if len(active_sales) > 1:
        stdev = np.std(active_sales, ddof=1)
        if stdev > 0:
            z_scores = np.abs((active_sales - np.mean(active_sales)) / stdev)
            anomaly_mask = z_scores >= 2.5
            if anomaly_mask.any():
                non_anomalous = active_sales[~anomaly_mask]
                if len(non_anomalous):
                    replacement_value = np.median(non_anomalous)
                else:
                    replacement_value = np.median(active_sales)
                active_sales = np.where(anomaly_mask, replacement_value, active_sales)

    return float(np.mean(active_sales))

df["avg_weekly_sales"] = df.apply(normalized_avg_weekly_sales, axis=1)
df["is_damaged"] = df["Condition"].eq("Damaged")
df["is_dead"] = (~df["is_damaged"]) & (((df["Inventory_Qty"] / np.maximum(df["avg_weekly_sales"], 0.1)) * 7) > 730)
df["is_protected"] = (df["Category"].eq("Home Automation")) | (df["Launch_Date"] >= RECENCY_CUTOFF)

df["effective_margin"] = np.where(df["is_damaged"] | df["is_dead"], 0.0, df["Inventory_Qty"] * df["Unit_Margin"])
df["effective_volume"] = df["Inventory_Qty"] * df["Unit_Volume"]

print("Total rows:", len(df))
print("Protected rows:", int(df["is_protected"].sum()))
print("Eligible rows:", int((~df["is_protected"]).sum()))
print("Damaged rows:", int(df["is_damaged"].sum()))
print("Dead rows:", int(df["is_dead"].sum()))
display(df[["SKU", "Category", "Condition", "Launch_Date", "avg_weekly_sales", "is_damaged", "is_dead", "is_protected"]].head(12))

Total rows: 150
Protected rows: 38
Eligible rows: 112
Damaged rows: 12
Dead rows: 64


,SKU,Category,Condition,Launch_Date,avg_weekly_sales,is_damaged,is_dead,is_protected
0,AUD126,Audio,New,2024-07-18,9.400000,False,False,False
1,AUD127,Audio,New,2024-07-30,27.375000,False,False,False
2,AUD128,Audio,New,2024-07-16,4.500000,False,False,False
3,AUD129,Audio,New,2024-07-28,19.875000,False,False,False
4,AUD130,Audio,New,2024-07-19,18.000000,False,False,False
5,AUD131,Audio,New,2024-07-16,4.000000,False,True,False
6,AUD132,Audio,New,2024-07-28,19.500000,False,False,False
7,AUD133,Audio,Damaged,2024-07-24,1.111111,True,False,False
8,AUD134,Audio,Damaged,2024-08-04,2.000000,True,False,True
9,AUD135,Audio,New,2024-08-01,25.500000,False,False,True


## Build the constrained optimization model

In [33]:
eligible_df = df.loc[~df["is_protected"]].copy().reset_index(drop=True)
assert len(eligible_df) >= 100, f"Need at least 100 eligible SKUs, found {len(eligible_df)}"

damaged = eligible_df["is_damaged"].astype(int).to_numpy()
dead = eligible_df["is_dead"].astype(int).to_numpy()
margin = eligible_df["effective_margin"].to_numpy(dtype=float)
volume = eligible_df["effective_volume"].to_numpy(dtype=float)

eligible_skus = set(eligible_df["SKU"])
sku_to_position = {sku: idx for idx, sku in enumerate(eligible_df["SKU"])}

dependency_edges = []
for _, row in eligible_df.iterrows():
    parent = str(row["Requires_Base_SKU"]).strip()
    if parent and parent in eligible_skus:
        dependency_edges.append((sku_to_position[parent], sku_to_position[row["SKU"]]))

print("Eligible SKUs:", len(eligible_df))
print("Dependency edges:", len(dependency_edges))

Eligible SKUs: 112
Dependency edges: 48


In [34]:
def solve_problem(cost_vector: np.ndarray, target_count: int, damaged_exact=None, dead_exact=None):
    rows = [np.ones(len(eligible_df))]
    lower = [target_count]
    upper = [target_count]

    if damaged_exact is not None:
        rows.append(damaged.astype(float))
        lower.append(damaged_exact)
        upper.append(damaged_exact)

    if dead_exact is not None:
        rows.append(dead.astype(float))
        lower.append(dead_exact)
        upper.append(dead_exact)

    for parent_idx, child_idx in dependency_edges:
        row = np.zeros(len(eligible_df))
        row[child_idx] = 1.0
        row[parent_idx] = -1.0
        rows.append(row)
        lower.append(-np.inf)
        upper.append(0.0)

    constraints = LinearConstraint(
        np.vstack(rows),
        np.array(lower),
        np.array(upper),
    )

    result = milp(
        c=cost_vector,
        constraints=constraints,
        integrality=np.ones(len(eligible_df), dtype=int),
        bounds=Bounds(np.zeros(len(eligible_df)), np.ones(len(eligible_df))),
    )

    if result.status != 0:
        raise RuntimeError(
            f"MILP failed for target={target_count}, damaged_exact={damaged_exact}, dead_exact={dead_exact}: {result.message}"
        )

    return result.x > 0.5


def solve_density_optimal(target_count: int):
    damaged_solution = solve_problem(-damaged.astype(float), target_count)
    optimal_damaged = int(damaged[damaged_solution].sum())

    dead_solution = solve_problem(-dead.astype(float), target_count, damaged_exact=optimal_damaged)
    optimal_dead = int(dead[dead_solution].sum())

    best_density = 0.0
    best_solution = None

    for _ in range(50):
        solution = solve_problem(
            margin - best_density * volume,
            target_count,
            damaged_exact=optimal_damaged,
            dead_exact=optimal_dead,
        )

        selected_margin = float(margin[solution].sum())
        selected_volume = float(volume[solution].sum())
        updated_density = selected_margin / selected_volume if selected_volume > 0 else 0.0

        best_solution = solution
        if abs(updated_density - best_density) < 1e-10:
            best_density = updated_density
            break
        best_density = updated_density

    selected = eligible_df.loc[best_solution].copy()
    return {
        "selected": selected,
        "optimal_damaged": optimal_damaged,
        "optimal_dead": optimal_dead,
        "density": best_density,
        "margin": float(selected["effective_margin"].sum()),
        "volume": float(selected["effective_volume"].sum()),
    }


def build_ranked_output(selected_df: pd.DataFrame) -> pd.DataFrame:
    ranked = selected_df.copy()

    ranked["priority_tier"] = np.select(
        [ranked["is_damaged"], ranked["is_dead"]],
        [0, 1],
        default=2,
    )
    ranked = ranked.sort_values(
        by=["priority_tier", "effective_volume", "SKU"],
        ascending=[True, False, True],
    ).reset_index(drop=True)
    ranked.insert(0, "Priority_Rank", np.arange(1, len(ranked) + 1))
    return ranked[["Priority_Rank", "SKU", "Product_Name"]]

## Solve both list sizes

In [35]:
results = {}

for target in (50, 100):
    result = solve_density_optimal(target)
    ranked_output = build_ranked_output(result["selected"])

    output_path = WORKSPACE_DIR / f"liquidation_list_{target}.csv"
    ranked_output.to_csv(output_path, index=False)

    results[target] = {
        **result,
        "ranked_output": ranked_output,
    }

    print(f"Target {target}")
    print("Damaged selected:", result["optimal_damaged"])
    print("Dead selected:", result["optimal_dead"])
    print("Density:", round(result["density"], 6))
    display(ranked_output.head(10))

Target 50
Damaged selected: 10
Dead selected: 38
Density: 0.198359


,Priority_Rank,SKU,Product_Name
0,1,CBL084,USB-C Cable 19
1,2,CHG039,Fast Charger 9
2,3,CSE110,Protective Case 10
3,4,CSE119,Case Sleeve 4
4,5,CHG040,Fast Charger 10
5,6,CSE106,Protective Case 6
6,7,AUD145,Eartip Pack 5
7,8,AUD133,Audio Buds 8
8,9,CSE121,Case Sleeve 6
9,10,CHG047,Fast Charger 17


Target 100
Damaged selected: 10
Dead selected: 58
Density: 3.531469


,Priority_Rank,SKU,Product_Name
0,1,CBL084,USB-C Cable 19
1,2,CHG039,Fast Charger 9
2,3,CSE110,Protective Case 10
3,4,CSE119,Case Sleeve 4
4,5,CHG040,Fast Charger 10
5,6,CSE106,Protective Case 6
6,7,AUD145,Eartip Pack 5
7,8,AUD133,Audio Buds 8
8,9,CSE121,Case Sleeve 6
9,10,CHG047,Fast Charger 17


## Validate outputs and save final report

In [36]:
for target in (50, 100):
    selected_skus = set(results[target]["selected"]["SKU"])
    protected_skus = set(df.loc[df["is_protected"], "SKU"])
    assert not (selected_skus & protected_skus)
    assert len(selected_skus) == target

    # Dependency consistency
    for _, row in results[target]["selected"].iterrows():
        parent = str(row["Requires_Base_SKU"]).strip()
        if parent and parent != "nan":
            assert parent in selected_skus

final_report = pd.DataFrame([{
    "reclaimed_space_100": results[100]["volume"],
    "total_margin_density_100": results[100]["density"],
    "total_margin_100": results[100]["margin"],
    "reclaimed_space_50": results[50]["volume"],
    "total_margin_density_50": results[50]["density"],
    "total_margin_50": results[50]["margin"],
}])

final_report_path = WORKSPACE_DIR / "final_report.csv"
final_report.to_csv(final_report_path, index=False)

print("Saved:")
print(WORKSPACE_DIR / "liquidation_list_50.csv")
print(WORKSPACE_DIR / "liquidation_list_100.csv")
print(final_report_path)
display(final_report)

Saved:
D:\Turing\My Samples\warehouse_liquidation_task\liquidation_list_50.csv
D:\Turing\My Samples\warehouse_liquidation_task\liquidation_list_100.csv
D:\Turing\My Samples\warehouse_liquidation_task\final_report.csv


,reclaimed_space_100,total_margin_density_100,total_margin_100,reclaimed_space_50,total_margin_density_50,total_margin_50
0,149143.05,3.531469,526694.03,100968.09,0.198359,20027.93
